# Workflow 3

This notebook is the first workflow as descrbed in this google document - https://docs.google.com/document/d/1qbmAjRa2V-anxj63YFFHdnovbY_U9MRi0uMAa4ZoCNg/edit.

The notebook merges the predictions (adjacent tiles) from SAM and does accuracy assessment.

In [1]:
import sys
sys.path.append('../')
from utils import *

# import required libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import os, glob, math, json
from matplotlib import pyplot as plt
from copy import deepcopy
from rtree import index
from geopandas.tools import sjoin
from shapely.geometry import box
from concurrent.futures import ThreadPoolExecutor
import time
import geoplanar
from datetime import datetime

today = datetime.now().strftime("%y%m%d")


# change the working directory
main_dir = r"D:/2212_PlanetOrtho_FieldBoundary/ForPaper/vector/"
gt_file = r"D:\2212_PlanetOrtho_FieldBoundary\ForPaper\vector\GroundTruth\240805_GroundTruth_FieldBoundaries_PT_V2.shp"
os.chdir(main_dir)

## Merge predictions from different checkpoints

In [2]:
%%time

COMPACTNESS_THRESHOLD = 0.5
previous_output = '240814_Workflow2_Output_v1'
enhanced = False

# create folder
out_folder = os.path.join(main_dir, previous_output.replace('Workflow2', 'Workflow3'))
if not os.path.exists(out_folder):
    os.mkdir(out_folder)

# get all the shapefiles generated in workflow step 1
all_files = [os.path.split(file)[-1] for file in glob.glob(f'{previous_output}/*V2.gpkg')]

# filter in/ out enhanced output
if enhanced:
    all_files = [
        file
        for file in all_files
        if 'Enhanced' in file
    ]
else:
    all_files = [
        file
        for file in all_files
        if not 'Enhanced' in file
    ]


# define a dataframe to store metrics
main_df = pd.DataFrame(columns=['File', 'Precision', 'Recall', 'F1-Score', 'Detection', 'IoU'])

# loop through all the wild cards
for epoch in ['T1', 'T2', 'T3', 'T4']:
    print(epoch)
    
    
    if enhanced:
        test_gdf = pd.concat([
            read_file_with_name(file) 
            for file in glob.glob(f'{previous_output}/*{epoch}*.gpkg')
            if 'Enhanced' in file
        ])
    else:
        test_gdf = pd.concat([
            read_file_with_name(file) 
            for file in glob.glob(f'{previous_output}/*{epoch}*.gpkg')
            if not 'Enhanced' in file
        ])
    
    test_gdf = test_gdf.explode(index_parts=True).reset_index(drop=True)
    test_gdf = test_gdf[['name', 'geometry']]
    test_gdf['ID_pred'] = test_gdf.index
    
    """
    After combining predictions, there could be overlapping polygons, deal with them
    """
    # take care of the polygons that are overlapped with their (almost!) duplicates
    test_gdf = overlap_resolution_single(test_gdf)
    
    
    # take care of the ones where small polygons are covered by large ones
    test_gdf = trim_overlaps(test_gdf, largest=True).explode(index_parts=True)
    test_gdf = test_gdf.loc[test_gdf.area > 100]
    
    # geoplanar polygon cutting creates weird shapes, get rid of those
    bounding_boxes = test_gdf.geometry.apply(lambda geom: geom.minimum_rotated_rectangle)
    test_gdf = test_gdf[test_gdf.area / bounding_boxes.area >= COMPACTNESS_THRESHOLD]
    
    # the previous step creates weird shapes, correct them
    test_gdf['geometry'] = test_gdf.buffer(-2, join_style=3).buffer(2, join_style=3)
    test_gdf = test_gdf.explode(index_parts=True)
    test_gdf = test_gdf.loc[test_gdf.area > 100]
    
    
    # export the combined file
    if enhanced:
        outfile1 = os.path.join(out_folder, f'{epoch}_Enhanced_segmented_Workflow3Merged.gpkg')
    else:
        outfile1 = os.path.join(out_folder, f'{epoch}_segmented_Workflow3Merged.gpkg')
    test_gdf.to_file(outfile1)

T1
T2
T3
T4
CPU times: total: 16.7 s
Wall time: 54.9 s


# Perform accuracy assessment

In [3]:
%%time

# get a list of all the files
files_list = glob.glob(f"{out_folder}/*.gpkg")

# create an empty DF to store metrics
metrics_df = pd.DataFrame(columns=['File', 'Precision', 'Recall', 'F1-Score', 'Detection', 'IoU'])

# loop through all the files and caculate metrics
for n, file in enumerate(files_list):
    print(os.path.split(file)[-1], f'{n}/{len(files_list)}..')
    
    """
    prepare the ground truth shapefile
    """
    gt_gdf = gpd.read_file(gt_file)
    gt_gdf['ID_gt'] = gt_gdf.index

    # add new cols in the ground truth gdf to calculate metrics later
    for new_col in ['max_overlap_id', 'iou', 'precision', 'recall', 'f1score']:
        gt_gdf[new_col] = -1 

    # remove empty geometries
    gt_gdf = gt_gdf.loc[
        (gt_gdf.geom_type.isin(['Polygon', 'MultiPolygon']))
    ]

    """
    read the predicted data
    """
    # read the predicted file
    pred_gdf = gpd.read_file(file)
    
    """
    calculate tota area based metrics
    """
    intersection_area = gt_gdf.dissolve().intersection(pred_gdf.dissolve()).area[0]
    union_area = gt_gdf.dissolve().union(pred_gdf.dissolve()).area[0]

    precision = round(intersection_area / pred_gdf.dissolve().area[0], 3)
    recall = round(intersection_area / gt_gdf.dissolve().area[0], 3)
    f1_score = round(2 * (precision * recall) / (precision + recall), 3)

    """
    calculate number of polygons detected
    """
    # Create spatial index for test layer for faster processing
    # can't craete this inside the function because it is unnecessary time wastage
    spatial_index = index.Index()
    for i, geometry in enumerate(pred_gdf.geometry):
        if len(geometry.bounds) > 0:
            spatial_index.insert(i, geometry.bounds)

    # for each polygon in the ground truth, find polygon in the test file that has IoU greater than 0.5
    gt_gdf = gt_gdf.apply(lambda row: wflow1_iou_threshold_metrics(row, gt_gdf, deepcopy(pred_gdf), spatial_index,
                                                                  threshold=0.5), axis=1)

    # calculate the precision for this test layer (how many polygons are identified)
    detection_percentage = round(100 * gt_gdf.loc[gt_gdf['max_overlap_id'] != -1].shape[0] / gt_gdf.shape[0], 3)
    detected_iou = round(gt_gdf.loc[gt_gdf['max_overlap_id'] != -1, 'iou'].mean(), 3)

    metrics_dict = {
            'File': [os.path.split(file)[-1]],
            'Precision': [precision],
            'Recall': [recall],
            'F1-Score': [f1_score],
            'Detection': [detection_percentage],
            'IoU': [detected_iou]
        }

    metrics_df = pd.concat([
        metrics_df, 
        pd.DataFrame(metrics_dict)
    ], axis=0)

    metrics_df.to_csv(f"D:/2212_PlanetOrtho_FieldBoundary/ForPaper/tables/{today}_Workflow3Metrics_v1.csv", index=False)

T1_Enhanced_segmented_Workflow3Merged.gpkg 0/8..


<timed exec>:69: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.


T1_segmented_Workflow3Merged.gpkg 1/8..
T2_Enhanced_segmented_Workflow3Merged.gpkg 2/8..
T2_segmented_Workflow3Merged.gpkg 3/8..
T3_Enhanced_segmented_Workflow3Merged.gpkg 4/8..
T3_segmented_Workflow3Merged.gpkg 5/8..
T4_Enhanced_segmented_Workflow3Merged.gpkg 6/8..
T4_segmented_Workflow3Merged.gpkg 7/8..
CPU times: total: 55 s
Wall time: 2min 49s
